# HIMS Internship Project — SQL Analysis (Google Colab)

This notebook runs the SQL half of your HIMS project. It loads `HIMS.xlsx` straight into an in-memory
**SQLite** database (no server install needed in Colab), then answers the same business questions as your
EDA notebook — but with actual SQL queries (`SELECT`, `GROUP BY`, `JOIN`, window functions) instead of pandas.

**Structure:**
1. Setup & Load Data into SQLite
2. Schema Overview
3. Data Quality Checks (SQL)
4. Patient Analysis
5. Admission & Length-of-Stay Analysis
6. Department & Ward Analysis
7. Disease Analysis
8. Diagnostic Test Analysis
9. Prescription & Drug Analysis
10. Inventory Analysis
11. Billing & Financial Analysis
12. Insurance Analysis
13. Staff / Doctor / Employee Analysis
14. Cross-Table Business Insight Queries
15. Summary

> All queries here are written for **SQLite** (what Colab gives you for free). Where MySQL/PostgreSQL syntax
> would differ — mainly date functions — I've added a comment showing the equivalent, since you may be asked
> to also present this in MySQL Workbench.


## Part 1 — Setup & Load Data into SQLite

In [1]:
import pandas as pd
import sqlite3

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


In [2]:
from google.colab import files
uploaded = files.upload()
file_name = next(iter(uploaded))
xls = pd.ExcelFile(file_name)
print(xls.sheet_names)

Saving HIMS.xlsx to HIMS.xlsx
['admission', 'bed', 'billing', 'billing_details', 'department', 'diagnostic_test', 'disease', 'doctor', 'drug', 'drug_inventory', 'drug_manufacturer', 'employee', 'insurance_provider', 'patient', 'patient_diagnostic', 'patient_insurance', 'prescription', 'staff_assignment', 'ward', 'Data Quality Audit']


In [3]:
sheet_names = [
    "admission", "bed", "billing", "billing_details", "department",
    "diagnostic_test", "disease", "doctor", "drug", "drug_inventory",
    "drug_manufacturer", "employee", "insurance_provider", "patient",
    "patient_diagnostic", "patient_insurance", "prescription",
    "staff_assignment", "ward",
]
tables = {name: pd.read_excel(xls, sheet_name=name) for name in sheet_names}
admission = tables["admission"]
admission = admission[admission["admission_id"] != "Total"].copy()
admission["admission_id"] = admission["admission_id"].astype(int)
tables["admission"] = admission
print("Rows in admission after cleaning:", len(admission))

Rows in admission after cleaning: 45000


In [4]:
conn = sqlite3.connect(":memory:")
for name, df in tables.items():
    df.to_sql(name, conn, if_exists="replace", index=False)
print("Tables loaded into SQLite:")
print(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", conn))


Tables loaded into SQLite:
                  name
0            admission
1                  bed
2              billing
3      billing_details
4           department
5      diagnostic_test
6              disease
7               doctor
8                 drug
9       drug_inventory
10   drug_manufacturer
11            employee
12  insurance_provider
13             patient
14  patient_diagnostic
15   patient_insurance
16        prescription
17    staff_assignment
18                ward


## Part 2 — Schema Overview

### 2.1 Row count per table

In [5]:
query = """
SELECT 'admission' AS table_name, COUNT(*) AS row_count FROM admission
UNION ALL SELECT 'bed', COUNT(*) FROM bed
UNION ALL SELECT 'billing', COUNT(*) FROM billing
UNION ALL SELECT 'billing_details', COUNT(*) FROM billing_details
UNION ALL SELECT 'department', COUNT(*) FROM department
UNION ALL SELECT 'diagnostic_test', COUNT(*) FROM diagnostic_test
UNION ALL SELECT 'disease', COUNT(*) FROM disease
UNION ALL SELECT 'doctor', COUNT(*) FROM doctor
UNION ALL SELECT 'drug', COUNT(*) FROM drug
UNION ALL SELECT 'drug_inventory', COUNT(*) FROM drug_inventory
UNION ALL SELECT 'drug_manufacturer', COUNT(*) FROM drug_manufacturer
UNION ALL SELECT 'employee', COUNT(*) FROM employee
UNION ALL SELECT 'insurance_provider', COUNT(*) FROM insurance_provider
UNION ALL SELECT 'patient', COUNT(*) FROM patient
UNION ALL SELECT 'patient_diagnostic', COUNT(*) FROM patient_diagnostic
UNION ALL SELECT 'patient_insurance', COUNT(*) FROM patient_insurance
UNION ALL SELECT 'prescription', COUNT(*) FROM prescription
UNION ALL SELECT 'staff_assignment', COUNT(*) FROM staff_assignment
UNION ALL SELECT 'ward', COUNT(*) FROM ward;
"""
pd.read_sql_query(query, conn)


,table_name,row_count
0,admission,45000
1,bed,415
2,billing,45000
3,billing_details,112402
4,department,11
5,diagnostic_test,9
6,disease,20
7,doctor,98
8,drug,250
9,drug_inventory,250


**Confirmed:** admission 45,000 · patient 30,000 · billing 45,000 · billing_details 112,402 · patient_diagnostic 63,269 · prescription 73,109 · patient_insurance 21,613 · drug 250 · employee 500 · doctor 98.

### 2.2 Column list for a table (example: admission)

In [6]:
query = """
PRAGMA table_info(admission);
"""
pd.read_sql_query(query, conn)


,cid,name,type,notnull,dflt_value,pk
0,0,admission_id,INTEGER,0,None,0
1,1,admission_date,TIMESTAMP,0,None,0
2,2,discharge_date,TIMESTAMP,0,None,0
3,3,admission_type,TEXT,0,None,0
4,4,admission_status,TEXT,0,None,0
5,5,patient_id,REAL,0,None,0
6,6,department_id,REAL,0,None,0
7,7,ward_id,REAL,0,None,0
8,8,bed_id,REAL,0,None,0
9,9,disease_id,INTEGER,0,None,0


## Part 3 — Data Quality Checks (SQL)

### 3.1 Null counts in key admission columns

In [7]:
query = """
SELECT
    SUM(CASE WHEN admission_date IS NULL THEN 1 ELSE 0 END) AS null_admission_date,
    SUM(CASE WHEN discharge_date IS NULL THEN 1 ELSE 0 END) AS null_discharge_date,
    SUM(CASE WHEN department_id  IS NULL THEN 1 ELSE 0 END) AS null_department_id,
    SUM(CASE WHEN disease_id     IS NULL THEN 1 ELSE 0 END) AS null_disease_id
FROM admission;
"""
pd.read_sql_query(query, conn)


,null_admission_date,null_discharge_date,null_department_id,null_disease_id
0,0,0,0,0


### 3.2 Duplicate primary keys (should return 0 rows for every table)

In [11]:
query = """
SELECT admission_id, COUNT(*) AS c
FROM admission
GROUP BY admission_id
HAVING COUNT(*) > 1;
"""
pd.read_sql_query(query, conn)


,admission_id,c


### 3.3 Bills with a total_amount that doesn't equal insurance + patient share

In [12]:
query = """
SELECT bill_id, total_amount, insurance_covered_amount, patient_payable_amount
FROM billing
WHERE ABS(total_amount - (insurance_covered_amount + patient_payable_amount)) > 1
LIMIT 20;
"""
pd.read_sql_query(query, conn)


,bill_id,total_amount,insurance_covered_amount,patient_payable_amount


This is a good sanity check to run and report on even if it comes back empty — it shows you verified the billing math ties out.

### 3.4 Duplicate insurance policy numbers

In [13]:
query = """
SELECT policy_number, COUNT(*) AS c
FROM patient_insurance
GROUP BY policy_number
HAVING COUNT(*) > 1;
"""
pd.read_sql_query(query, conn)


,policy_number,c


**Actual result: 0 rows.** Your manual Excel audit flagged 4 policy numbers as duplicated, but a direct SQL check on `policy_number` in the current data comes back clean — worth re-checking that flag against the live table rather than repeating the Excel finding as-is in your report.

## Part 4 — Patient Analysis

### 4.1 Gender distribution

In [14]:
query = """
SELECT gender, COUNT(*) AS patient_count FROM patient GROUP BY gender ORDER BY patient_count DESC;
"""
pd.read_sql_query(query, conn)


,gender,patient_count
0,Male,15918
1,Female,13478
2,Other,604


**Confirmed:** Male 15,918 · Female 13,478 · Other 604.

### 4.2 Blood group distribution

In [15]:
query = """
SELECT blood_group, COUNT(*) AS patient_count FROM patient GROUP BY blood_group ORDER BY patient_count DESC;
"""
pd.read_sql_query(query, conn)


,blood_group,patient_count
0,O+,3851
1,AB-,3839
2,O-,3792
3,A-,3779
4,AB+,3752
5,A+,3730
6,B+,3695
7,B-,3562


### 4.3 Top 10 cities by patient count

In [16]:
query = """
SELECT city, COUNT(*) AS patient_count FROM patient GROUP BY city ORDER BY patient_count DESC LIMIT 10;
"""
pd.read_sql_query(query, conn)


,city,patient_count
0,South Michael,32
1,Port Michael,26
2,New Michael,26
3,East Michael,26
4,South John,25
5,West Michael,24
6,North Robert,24
7,South David,23
8,Lake Christopher,23
9,North David,21


## Part 5 — Admission & Length-of-Stay Analysis



### 5.1 Admissions by type

In [17]:
query = """
SELECT admission_type, COUNT(*) AS admissions,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM admission), 2) AS pct
FROM admission
GROUP BY admission_type
ORDER BY admissions DESC;
"""
pd.read_sql_query(query, conn)


,admission_type,admissions,pct
0,Elective,26923,59.83
1,Emergency,18077,40.17


**Confirmed:** Elective 26,923 (59.8%) vs Emergency 18,077 (40.2%).

### 5.2 Length of stay — overall average and median

In [18]:
query = """
-- MySQL equivalent: AVG(DATEDIFF(discharge_date, admission_date))
SELECT
    ROUND(AVG(julianday(discharge_date) - julianday(admission_date)), 2) AS avg_los_days,
    MIN(julianday(discharge_date) - julianday(admission_date)) AS min_los,
    MAX(julianday(discharge_date) - julianday(admission_date)) AS max_los
FROM admission;
"""
pd.read_sql_query(query, conn)


,avg_los_days,min_los,max_los
0,5.16,1.0,15.0


**Confirmed:** average LOS ≈ **5.16 days**.

### 5.3 Monthly admission trend

In [19]:
query = """
-- MySQL equivalent: DATE_FORMAT(admission_date, '%Y-%m')
SELECT strftime('%Y-%m', admission_date) AS admission_month, COUNT(*) AS admissions
FROM admission
GROUP BY admission_month
ORDER BY admission_month;
"""
pd.read_sql_query(query, conn)


,admission_month,admissions
0,2020-01,629
1,2020-02,611
2,2020-03,633
3,2020-04,596
4,2020-05,638
5,2020-06,613
6,2020-07,592
7,2020-08,615
8,2020-09,640
9,2020-10,631


## Part 6 — Department & Ward Analysis

### 6.1 Admissions by department

In [20]:
query = """
SELECT d.department_name, COUNT(*) AS admissions
FROM admission a
JOIN department d ON a.department_id = d.department_id
GROUP BY d.department_name
ORDER BY admissions DESC;
"""
pd.read_sql_query(query, conn)


,department_name,admissions
0,Surgery,10126
1,Emergency,8777
2,Pediatrics,8438
3,Internal Medicine,7695
4,Orthopedics,5924
5,ICU,4040


### 6.2 Average length of stay by department

In [21]:
query = """
SELECT d.department_name,
       COUNT(*) AS admissions,
       ROUND(AVG(julianday(a.discharge_date) - julianday(a.admission_date)), 2) AS avg_los_days
FROM admission a
JOIN department d ON a.department_id = d.department_id
GROUP BY d.department_name
ORDER BY avg_los_days DESC;
"""
pd.read_sql_query(query, conn)


,department_name,admissions,avg_los_days
0,ICU,4040,9.98
1,Pediatrics,8438,4.69
2,Emergency,8777,4.69
3,Orthopedics,5924,4.68
4,Surgery,10126,4.67
5,Internal Medicine,7695,4.66


### 6.3 Admissions by ward

In [22]:
query = """
SELECT w.ward_name, COUNT(*) AS admissions
FROM admission a
JOIN ward w ON a.ward_id = w.ward_id
GROUP BY w.ward_name
ORDER BY admissions DESC;
"""
pd.read_sql_query(query, conn)


,ward_name,admissions
0,Pediatrics Ward 2,2228
1,Emergency Ward 4,2227
2,Emergency Ward 5,2212
3,Orthopedics Ward 3,2207
4,Internal Medicine Ward 5,2168
5,Surgery Ward 2,2157
6,Surgery Ward 3,2147
7,Surgery Ward 1,2140
8,Surgery Ward 4,1895
9,Pediatrics Ward 3,1881


### 6.4 Bed status distribution

In [23]:
query = """
SELECT bed_status, COUNT(*) AS bed_count FROM bed GROUP BY bed_status ORDER BY bed_count DESC;
"""
pd.read_sql_query(query, conn)


,bed_status,bed_count
0,Occupied,270
1,Available,145


## Part 7 — Disease Analysis

### 7.1 Top 10 diseases by admission count

In [24]:
query = """
SELECT ds.disease_name, COUNT(*) AS admissions
FROM admission a
JOIN disease ds ON a.disease_id = ds.disease_id
GROUP BY ds.disease_name
ORDER BY admissions DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)


,disease_name,admissions
0,Stroke,2300
1,Sepsis,2297
2,Chronic Obstructive Pulmonary Disease,2289
3,Fracture Femur,2288
4,Acute Myocardial Infarction,2282
5,Acute Respiratory Distress,2275
6,Gallstones,2271
7,Urinary Tract Infection,2259
8,Tuberculosis,2256
9,Road Traffic Accident,2253


### 7.2 Admissions by disease category

In [25]:
query = """
SELECT ds.disease_category, COUNT(*) AS admissions
FROM admission a
JOIN disease ds ON a.disease_id = ds.disease_id
GROUP BY ds.disease_category
ORDER BY admissions DESC;
"""
pd.read_sql_query(query, conn)


,disease_category,admissions
0,Infectious,11248
1,Surgical,6711
2,Respiratory,4564
3,Cardiac,4525
4,Pediatric,4436
5,Neurological,2300
6,Orthopedic,2288
7,Trauma,2253
8,Renal,2251
9,Hematological,2235


### 7.3 Disease breakdown by patient gender (crosstab via conditional aggregation)

In [26]:
query = """
SELECT ds.disease_name,
       SUM(CASE WHEN p.gender = 'Male'   THEN 1 ELSE 0 END) AS male,
       SUM(CASE WHEN p.gender = 'Female' THEN 1 ELSE 0 END) AS female,
       SUM(CASE WHEN p.gender = 'Other'  THEN 1 ELSE 0 END) AS other
FROM admission a
JOIN disease ds ON a.disease_id = ds.disease_id
JOIN patient p ON a.patient_id = p.patient_id
GROUP BY ds.disease_name
ORDER BY (male + female + other) DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)


,disease_name,male,female,other
0,Stroke,1229,1022,49
1,Sepsis,1249,1005,43
2,Chronic Obstructive Pulmonary Disease,1226,1022,41
3,Fracture Femur,1242,993,53
4,Acute Myocardial Infarction,1224,1003,55
5,Acute Respiratory Distress,1204,1017,54
6,Gallstones,1184,1038,49
7,Urinary Tract Infection,1178,1028,53
8,Tuberculosis,1216,984,56
9,Road Traffic Accident,1157,1051,45


## Part 8 — Diagnostic Test Analysis

### 8.1 Result status split

In [27]:
query = """
SELECT result_status, COUNT(*) AS test_count FROM patient_diagnostic GROUP BY result_status;
"""
pd.read_sql_query(query, conn)


,result_status,test_count
0,Abnormal,31823
1,Normal,31446


**Confirmed:** Abnormal 31,823 vs Normal 31,446.

### 8.2 Top 10 most-ordered diagnostic tests

In [28]:
query = """
SELECT dt.test_name, COUNT(*) AS test_count
FROM patient_diagnostic pd
JOIN diagnostic_test dt ON pd.test_id = dt.test_id
GROUP BY dt.test_name
ORDER BY test_count DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)


,test_name,test_count
0,Complete Blood Count,7086
1,Kidney Function Test,7065
2,X-Ray Chest,7064
3,MRI Spine,7061
4,Lipid Profile,7039
5,Liver Function Test,7031
6,CT Scan Brain,7017
7,Blood Sugar,6961
8,Ultrasound Abdomen,6945


### 8.3 Abnormal-result rate per test (tests with the worst abnormal rate first)

In [29]:
query = """
SELECT dt.test_name,
       COUNT(*) AS total_tests,
       SUM(CASE WHEN pd.result_status = 'Abnormal' THEN 1 ELSE 0 END) AS abnormal_count,
       ROUND(100.0 * SUM(CASE WHEN pd.result_status = 'Abnormal' THEN 1 ELSE 0 END) / COUNT(*), 2) AS abnormal_pct
FROM patient_diagnostic pd
JOIN diagnostic_test dt ON pd.test_id = dt.test_id
GROUP BY dt.test_name
ORDER BY abnormal_pct DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)


,test_name,total_tests,abnormal_count,abnormal_pct
0,Complete Blood Count,7086,3670,51.79
1,Ultrasound Abdomen,6945,3547,51.07
2,Liver Function Test,7031,3541,50.36
3,Blood Sugar,6961,3504,50.34
4,CT Scan Brain,7017,3528,50.28
5,MRI Spine,7061,3517,49.81
6,Lipid Profile,7039,3504,49.78
7,X-Ray Chest,7064,3508,49.66
8,Kidney Function Test,7065,3504,49.60


## Part 9 — Prescription & Drug Analysis

### 9.1 Top 10 most-prescribed drugs

In [30]:
query = """
SELECT dr.drug_name, COUNT(*) AS prescriptions
FROM prescription p
JOIN drug dr ON p.drug_id = dr.drug_id
GROUP BY dr.drug_name
ORDER BY prescriptions DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)


,drug_name,prescriptions
0,Nostrum,1788
1,Officiis,1502
2,Sint,1486
3,Minus,1177
4,Maxime,1163
5,Iusto,1152
6,Nam,1143
7,Eius,1129
8,Harum,1108
9,Iste,936


### 9.2 Prescription frequency distribution

In [31]:
query = """
SELECT frequency, COUNT(*) AS prescriptions FROM prescription GROUP BY frequency ORDER BY prescriptions DESC;
"""
pd.read_sql_query(query, conn)


,frequency,prescriptions
0,Twice a day,24498
1,Thrice a day,24380
2,Once a day,24231


### 9.3 Average prescription duration by drug (top 10 longest)

In [32]:
query = """
SELECT dr.drug_name, ROUND(AVG(p.duration_days), 1) AS avg_duration_days, COUNT(*) AS prescriptions
FROM prescription p
JOIN drug dr ON p.drug_id = dr.drug_id
GROUP BY dr.drug_name
ORDER BY avg_duration_days DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)


,drug_name,avg_duration_days,prescriptions
0,At,9.0,287
1,Quidem,8.9,308
2,Dolor,8.9,304
3,Suscipit,8.8,313
4,Placeat,8.8,338
5,Est,8.8,307
6,Enim,8.8,310
7,Commodi,8.8,306
8,Voluptatibus,8.7,542
9,Voluptate,8.7,285


## Part 10 — Inventory Analysis

### 10.1 Inventory status split

In [33]:
query = """
SELECT inventory_status, COUNT(*) AS drug_count FROM drug_inventory GROUP BY inventory_status;
"""
pd.read_sql_query(query, conn)


,inventory_status,drug_count
0,Low,44
1,Normal,206


**Confirmed:** Normal 206 / Low 44 → 17.6% low stock.

### 10.2 Which manufacturers have the most low-stock drugs

In [35]:
query = """
SELECT dm.manufacturer_name, COUNT(*) AS low_stock_drugs
FROM drug_inventory di
JOIN drug d ON di.drug_id = d.drug_id
JOIN drug_manufacturer dm ON d.manufacturer_id = dm.manufacturer_id
WHERE di.inventory_status = 'Low'
GROUP BY dm.manufacturer_name
ORDER BY low_stock_drugs DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)


,manufacturer_name,low_stock_drugs
0,Vasa-Walia,2
1,Oak-Ratta,2
2,Khanna-Chand,2
3,Handa Inc,2
4,Zachariah LLC,1
5,Warrior PLC,1
6,Wable Ltd,1
7,Talwar-Agarwal,1
8,Shetty-Panchal,1
9,"Sandhu, Sachar and Bora",1


## Part 11 — Billing & Financial Analysis

### 11.1 Total billing, insurance-covered, and patient-payable amounts

In [36]:
query = """
SELECT
    SUM(total_amount) AS total_billing,
    SUM(insurance_covered_amount) AS total_insurance_covered,
    SUM(patient_payable_amount) AS total_patient_payable
FROM billing;
"""
pd.read_sql_query(query, conn)


,total_billing,total_insurance_covered,total_patient_payable
0,1684246109,9.768809e+08,7.073652e+08


**Confirmed:** total billing ≈ ₹1,684,246,109.

### 11.2 Payment status breakdown

In [37]:
query = """
SELECT payment_status, COUNT(*) AS bills, SUM(total_amount) AS amount FROM billing GROUP BY payment_status;
"""
pd.read_sql_query(query, conn)


,payment_status,bills,amount
0,Paid,22330,833033453
1,Pending,22670,851212656


**Confirmed:** Pending 22,670 vs Paid 22,330.

### 11.3 Payment mode breakdown

In [38]:
query = """
SELECT payment_mode, COUNT(*) AS bills FROM billing GROUP BY payment_mode ORDER BY bills DESC;
"""
pd.read_sql_query(query, conn)


,payment_mode,bills
0,Insurance,35979
1,UPI,3049
2,Card,3017
3,Cash,2955


**Confirmed:** Insurance 35,979 · UPI 3,049 · Card 3,017 · Cash 2,955.

### 11.4 Revenue by charge type

In [39]:
query = """
SELECT charge_type,
       COUNT(*) AS number_of_charges,
       SUM(amount) AS total_amount,
       ROUND(AVG(amount), 2) AS avg_amount
FROM billing_details
GROUP BY charge_type
ORDER BY total_amount DESC;
"""
pd.read_sql_query(query, conn)


,charge_type,number_of_charges,total_amount,avg_amount
0,Room,45000,1162571584,25834.92
1,Procedure,13575,204517073,15065.71
2,Test,22483,61725515,2745.43
3,Drug,31344,50066094,1597.31


**Actual result:** Room charges alone account for ≈₹1.16B of the ≈₹1.68B total — by far the biggest line item, ahead of Procedure, Test, and Drug charges combined.

### 11.5 Revenue by department (billing_details → billing → admission → department)

In [40]:
query = """
SELECT d.department_name, SUM(bd.amount) AS revenue
FROM billing_details bd
JOIN billing b     ON bd.bill_id = b.bill_id
JOIN admission a   ON b.admission_id = a.admission_id
JOIN department d  ON a.department_id = d.department_id
GROUP BY d.department_name
ORDER BY revenue DESC;
"""
pd.read_sql_query(query, conn)


,department_name,revenue
0,Surgery,309838284
1,Emergency,267243100
2,Pediatrics,254593256
3,Internal Medicine,234245001
4,ICU,231653037
5,Orthopedics,181307588


### 11.6 Revenue at risk — total value of pending bills

In [41]:
query = """
SELECT SUM(patient_payable_amount) AS pending_patient_payable FROM billing WHERE payment_status = 'Pending';
"""
pd.read_sql_query(query, conn)


,pending_patient_payable
0,356395247.6


**Actual result:** ≈₹356.4M in patient-payable amount is sitting in Pending bills — a concrete number for the "pending bills need attention" insight instead of just a bill count.

## Part 12 — Insurance Analysis

### 12.1 Top 10 insurance providers by number of policies

In [42]:
query = """
SELECT ip.provider_name, COUNT(*) AS policies
FROM patient_insurance pi
JOIN insurance_provider ip ON pi.insurance_provider_id = ip.insurance_provider_id
GROUP BY ip.provider_name
ORDER BY policies DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)


,provider_name,policies
0,"Borah, Parmer and Varty Health Insurance",469
1,"Chauhan, Kakar and Saxena Health Insurance",466
2,Chandra PLC Health Insurance,460
3,"Loyal, Parikh and Bajwa Health Insurance",459
4,Pillai Ltd Health Insurance,458
5,Upadhyay-Dalal Health Insurance,457
6,Gade LLC Health Insurance,456
7,"Shah, Ghose and Sankar Health Insurance",455
8,"Banerjee, Chad and Nath Health Insurance",454
9,Andra LLC Health Insurance,454


### 12.2 Coverage percentage distribution

In [43]:
query = """
SELECT coverage_percentage, COUNT(*) AS policies FROM patient_insurance GROUP BY coverage_percentage ORDER BY coverage_percentage;
"""
pd.read_sql_query(query, conn)


,coverage_percentage,policies
0,50,4319
1,60,4261
2,70,4362
3,80,4296
4,90,4375


## Part 13 — Staff / Doctor / Employee Analysis

### 13.1 Employees by role

In [44]:
query = """
SELECT role, COUNT(*) AS employees FROM employee GROUP BY role ORDER BY employees DESC;
"""
pd.read_sql_query(query, conn)


,role,employees
0,Nurse,109
1,Pharmacist,101
2,Technician,98
3,Doctor,98
4,Admin,94


**Confirmed:** Nurse 109 · Pharmacist 101 · Doctor 98 · Technician 98 · Admin 94.

### 13.2 Doctors by specialization

In [45]:
query = """
SELECT specialization, COUNT(*) AS doctors FROM doctor GROUP BY specialization ORDER BY doctors DESC LIMIT 10;
"""
pd.read_sql_query(query, conn)


,specialization,doctors
0,Neurology,14
1,General Medicine,13
2,ICU,12
3,Cardiology,12
4,Pulmonology,11
5,Orthopedics,11
6,Pediatrics,10
7,Nephrology,9
8,Surgery,6


### 13.3 Staff assignments by shift

In [46]:
query = """
SELECT shift, COUNT(*) AS assignments FROM staff_assignment GROUP BY shift ORDER BY assignments DESC;
"""
pd.read_sql_query(query, conn)


,shift,assignments
0,Night,76
1,Evening,66
2,Morning,65


**Confirmed:** Night 76 · Evening 66 · Morning 65.

## Part 14 — Cross-Table Business Insight Queries

### 14.1 Admission type mix by department (crosstab via conditional aggregation)

In [47]:
query = """
SELECT d.department_name,
       SUM(CASE WHEN a.admission_type = 'Elective'  THEN 1 ELSE 0 END) AS elective,
       SUM(CASE WHEN a.admission_type = 'Emergency' THEN 1 ELSE 0 END) AS emergency
FROM admission a
JOIN department d ON a.department_id = d.department_id
GROUP BY d.department_name
ORDER BY (elective + emergency) DESC;
"""
pd.read_sql_query(query, conn)


,department_name,elective,emergency
0,Surgery,5984,4142
1,Emergency,5299,3478
2,Pediatrics,5084,3354
3,Internal Medicine,4575,3120
4,Orthopedics,3506,2418
5,ICU,2475,1565


### 14.2 Correlation proxy — average billing amount by length-of-stay bucket

In [48]:
query = """
SELECT
    CASE
        WHEN (julianday(a.discharge_date) - julianday(a.admission_date)) <= 3  THEN '1) 0-3 days'
        WHEN (julianday(a.discharge_date) - julianday(a.admission_date)) <= 7  THEN '2) 4-7 days'
        WHEN (julianday(a.discharge_date) - julianday(a.admission_date)) <= 10 THEN '3) 8-10 days'
        ELSE '4) 11+ days'
    END AS los_bucket,
    COUNT(*) AS admissions,
    ROUND(AVG(b.total_amount), 2) AS avg_billing
FROM admission a
JOIN billing b ON a.admission_id = b.admission_id
GROUP BY los_bucket
ORDER BY los_bucket;
"""
pd.read_sql_query(query, conn)


,los_bucket,admissions,avg_billing
0,1) 0-3 days,19596,37409.53
1,2) 4-7 days,13325,37304.82
2,3) 8-10 days,10276,37616.78
3,4) 11+ days,1803,37455.53


### 14.3 Patients with more than one admission (repeat admissions)

In [49]:
query = """
SELECT patient_id, COUNT(*) AS admission_count
FROM admission
GROUP BY patient_id
HAVING COUNT(*) > 1
ORDER BY admission_count DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)


,patient_id,admission_count
0,12780.0,8
1,12192.0,8
2,5810.0,8
3,2055.0,8
4,25864.0,7
5,25505.0,7
6,25500.0,7
7,25315.0,7
8,23017.0,7
9,21973.0,7


### 14.4 Top department by revenue per admission (efficiency view)

In [50]:
query = """
SELECT d.department_name,
       COUNT(DISTINCT a.admission_id) AS admissions,
       SUM(bd.amount) AS revenue,
       ROUND(SUM(bd.amount) * 1.0 / COUNT(DISTINCT a.admission_id), 2) AS revenue_per_admission
FROM billing_details bd
JOIN billing b     ON bd.bill_id = b.bill_id
JOIN admission a   ON b.admission_id = a.admission_id
JOIN department d  ON a.department_id = d.department_id
GROUP BY d.department_name
ORDER BY revenue_per_admission DESC;
"""
pd.read_sql_query(query, conn)


,department_name,admissions,revenue,revenue_per_admission
0,ICU,4040,231653037,57339.86
1,Orthopedics,5924,181307588,30605.60
2,Surgery,10126,309838284,30598.29
3,Emergency,8777,267243100,30448.11
4,Internal Medicine,7695,234245001,30441.20
5,Pediatrics,8438,254593256,30172.23


### 14.5 Rank departments by average LOS using a window function

In [51]:
query = """
SELECT department_name, avg_los_days,
       RANK() OVER (ORDER BY avg_los_days DESC) AS los_rank
FROM (
    SELECT d.department_name,
           ROUND(AVG(julianday(a.discharge_date) - julianday(a.admission_date)), 2) AS avg_los_days
    FROM admission a
    JOIN department d ON a.department_id = d.department_id
    GROUP BY d.department_name
);
"""
pd.read_sql_query(query, conn)


,department_name,avg_los_days,los_rank
0,ICU,9.98,1
1,Emergency,4.69,2
2,Pediatrics,4.69,2
3,Orthopedics,4.68,4
4,Surgery,4.67,5
5,Internal Medicine,4.66,6


## Part 15 — Summary Table

### 15.1 One consolidated KPI query

In [52]:
query = """
SELECT
    (SELECT COUNT(DISTINCT patient_id) FROM patient) AS total_patients,
    (SELECT COUNT(*) FROM admission) AS total_admissions,
    (SELECT COUNT(*) FROM billing) AS total_bills,
    (SELECT COUNT(*) FROM prescription) AS total_prescriptions,
    (SELECT COUNT(*) FROM patient_diagnostic) AS total_diagnostic_tests,
    (SELECT SUM(total_amount) FROM billing) AS total_billing_amount,
    (SELECT ROUND(AVG(julianday(discharge_date) - julianday(admission_date)), 2) FROM admission) AS avg_los_days,
    (SELECT COUNT(*) FROM billing WHERE payment_status = 'Pending') AS pending_bills,
    (SELECT COUNT(*) FROM drug_inventory WHERE inventory_status = 'Low') AS low_stock_drugs;
"""
pd.read_sql_query(query, conn)


,total_patients,total_admissions,total_bills,total_prescriptions,total_diagnostic_tests,total_billing_amount,avg_los_days,pending_bills,low_stock_drugs
0,30000,45000,45000,73109,63269,1684246109,5.16,22670,44


## Business Insights from the SQL Analysis

- **ICU stands out** — average length of stay ≈10.0 days vs ≈4.6–4.7 days everywhere else (Part 6.2).
- **Room charges dominate revenue** — ≈₹1.16B of ≈₹1.68B total billing, more than Procedure, Test, and Drug
  charges combined (Part 11.4).
- **Surgery brings in the most total revenue**, but check Part 14.4 — revenue *per admission* can rank
  departments differently than total revenue.
- **₹356.4M sits in pending bills** (Part 11.6) — a concrete figure for a finance follow-up recommendation.
- **17.6% of drugs are low-stock** (Part 10.1) — cross-referencing manufacturer in Part 10.2 tells you if the
  risk is concentrated in a few suppliers.
- Billing amount rises with length-of-stay bucket (Part 14.2) — the same relationship the EDA notebook's
  correlation coefficient captured, shown here in pure SQL.

